# Refinamiento Open Flights

In [1]:
from pyspark.sql import SparkSession, functions as F, types as T

spark = (
    SparkSession.builder
    .appName("refinamiento_openflights")
    .config("spark.driver.memory", "2g")
    .enableHiveSupport()
    .getOrCreate()
)

RAW = "/Obligatorio/landing/openflights"
REFINED = "/Obligatorio/refined"

airports_schema = T.StructType([
    T.StructField("airport_id",T.IntegerType()), T.StructField("name",T.StringType()),
    T.StructField("city",T.StringType()), T.StructField("country",T.StringType()),
    T.StructField("iata",T.StringType()), T.StructField("icao",T.StringType()),
    T.StructField("latitude",T.DoubleType()), T.StructField("longitude",T.DoubleType()),
    T.StructField("altitude",T.IntegerType()), T.StructField("timezone",T.DoubleType()),
    T.StructField("dst",T.StringType()), T.StructField("tz_database",T.StringType()),
    T.StructField("type",T.StringType()), T.StructField("source",T.StringType()),
])
airlines_schema = T.StructType([
    T.StructField("airline_id",T.IntegerType()), T.StructField("name",T.StringType()),
    T.StructField("alias",T.StringType()), T.StructField("iata",T.StringType()),
    T.StructField("icao",T.StringType()), T.StructField("callsign",T.StringType()),
    T.StructField("country",T.StringType()), T.StructField("active",T.StringType()),
])
routes_schema = T.StructType([
    T.StructField("airline",T.StringType()), T.StructField("airline_id",T.IntegerType()),
    T.StructField("source_airport",T.StringType()), T.StructField("source_airport_id",T.IntegerType()),
    T.StructField("dest_airport",T.StringType()), T.StructField("dest_airport_id",T.IntegerType()),
    T.StructField("codeshare",T.StringType()), T.StructField("stops",T.IntegerType()),
    T.StructField("equipment",T.StringType()),
])

airports_raw = spark.read.option("header",True).schema(airports_schema).csv(f"{RAW}/airports.csv")
airlines_raw = spark.read.option("header",True).schema(airlines_schema).csv(f"{RAW}/airlines.csv")
routes_raw   = spark.read.option("header",True).schema(routes_schema).csv(f"{RAW}/routes.csv")

# Helpers de limpieza
MARC = ["", "\\N", "-", "N/A", "NA"]
def limpiar(c):
    s = F.trim(F.col(c).cast("string"))
    return F.when(s.isin(MARC), None).otherwise(s)
def norm(c):
    # normaliza texto para generar claves determinísticas
    return F.lower(F.trim(F.regexp_replace(F.col(c), r"\s+", " ")))
def country_key(c):
    return F.sha2(norm(c), 256)




Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


2026-06-23T23:52:53,573 WARN [Thread-4] org.apache.hadoop.util.NativeCodeLoader - Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
# AEROPUERTOS: limpiar marcadores, normalizar códigos, filtrar type='airport'
airports_clean = (
    airports_raw
    .withColumn("name", limpiar("name"))
    .withColumn("city", limpiar("city"))
    .withColumn("country", limpiar("country"))
    .withColumn("iata", F.upper(limpiar("iata")))
    .withColumn("icao", F.upper(limpiar("icao")))
    .withColumn("type", limpiar("type"))
    .filter(F.col("type") == "airport")
    .dropDuplicates(["airport_id"])
    .withColumn("country_id", country_key("country"))
    .withColumn("city_id", F.sha2(F.concat_ws("|", norm("country"), norm("city")), 256))
)
print("Aeropuertos (type=airport):", airports_clean.count())

# AEROLÍNEAS: normalizar 'active' (incluida la 'n' minúscula) a booleano
airlines_clean = (
    airlines_raw
    .withColumn("name", limpiar("name"))
    .withColumn("country", limpiar("country"))
    .withColumn("iata", F.upper(limpiar("iata")))
    .withColumn("icao", F.upper(limpiar("icao")))
    .withColumn("active_bool", F.upper(F.trim(F.col("active"))) == "Y")
    .dropDuplicates(["airline_id"])
    .withColumn("country_id", country_key("country"))
)
print("Aerolíneas:", airlines_clean.count())

# RUTAS: tipar/normalizar, marcar codeshare y directas, quitar origen=destino,
# deduplicar por clave compuesta y validar integridad contra airports/airlines refinados.
ap_ids = airports_clean.select("airport_id")
al_ids = airlines_clean.select("airline_id")

routes_clean = (
    routes_raw
    .withColumn("equipment", limpiar("equipment"))
    .withColumn("is_codeshare", limpiar("codeshare").isNotNull())
    .withColumn("is_direct", F.col("stops") == 0)
    .filter(F.col("airline_id").isNotNull() &
            F.col("source_airport_id").isNotNull() &
            F.col("dest_airport_id").isNotNull())
    .filter(F.col("source_airport_id") != F.col("dest_airport_id"))   # quita la ruta origen=destino
    .dropDuplicates(["airline_id","source_airport_id","dest_airport_id","equipment"])
    .join(ap_ids.withColumnRenamed("airport_id","source_airport_id"), "source_airport_id", "left_semi")
    .join(ap_ids.withColumnRenamed("airport_id","dest_airport_id"), "dest_airport_id", "left_semi")
    .join(al_ids, "airline_id", "left_semi")
)
print("Rutas crudas:", routes_raw.count(), "-> rutas refinadas:", routes_clean.count())
print("Rutas descartadas:", routes_raw.count() - routes_clean.count())



Aeropuertos (type=airport): 8264
Aerolíneas: 6162


[Stage 21:>                                                         (0 + 1) / 1]

Rutas crudas: 67663 -> rutas refinadas: 66707


Rutas descartadas: 956


In [3]:
# country_id determinístico (sha2 del nombre normalizado). country_iso no viene en OpenFlights.
paises = (
    airports_clean.select("country").union(airlines_clean.select("country"))
    .filter(F.col("country").isNotNull())
    .withColumn("country_id", country_key("country"))
)
dim_country = (
    paises.groupBy("country_id")
    .agg(F.first("country").alias("country_name"))
    .withColumn("country_iso", F.lit(None).cast("string"))
    .withColumn("source", F.lit("openflights"))
    .select("country_id","country_name","country_iso","source")
)
print("dim_country (países distintos):", dim_country.count())

dim_country.write.mode("overwrite").parquet(f"{REFINED}/dim_country")



dim_country (países distintos): 314


In [4]:
dim_airport = (
    airports_clean.select(
        "airport_id",
        F.col("name").alias("airport_name"),
        F.col("iata").alias("iata_code"),
        F.col("icao").alias("icao_code"),
        "city_id",
        "country_id",
        "latitude",
        "longitude",
        "timezone",
    )
)
print("dim_airport:", dim_airport.count())
dim_airport.write.mode("overwrite").parquet(f"{REFINED}/dim_airport")



dim_airport: 8264


In [5]:
dim_airline = (
    airlines_clean.select(
        "airline_id",
        F.col("name").alias("airline_name"),
        F.col("iata").alias("iata_code"),
        F.col("icao").alias("icao_code"),
        "country_id",
        F.col("active_bool").alias("active"),
    )
)
print("dim_airline:", dim_airline.count())
dim_airline.write.mode("overwrite").parquet(f"{REFINED}/dim_airline")



dim_airline: 6162


In [6]:
# País de origen/destino de cada aeropuerto (para enriquecer la ruta)
ap_country = airports_clean.select(F.col("airport_id"), F.col("country_id"))

# Granularidad del modelo: ruta (aerolínea, origen, destino). Colapsamos equipment
# y contamos las variantes en route_count.
fact_air_route = (
    routes_clean
    .groupBy("airline_id","source_airport_id","dest_airport_id")
    .agg(
        F.max("stops").alias("stops"),
        F.concat_ws("|", F.sort_array(F.collect_set("equipment"))).alias("equipment"),
        F.count("*").alias("route_count"),
    )
    .join(ap_country.withColumnRenamed("airport_id","source_airport_id")
                    .withColumnRenamed("country_id","source_country_id"),
          "source_airport_id", "left")
    .join(ap_country.withColumnRenamed("airport_id","dest_airport_id")
                    .withColumnRenamed("country_id","destination_country_id"),
          "dest_airport_id", "left")
    .select(
        "airline_id",
        "source_airport_id",
        F.col("dest_airport_id").alias("destination_airport_id"),
        "source_country_id",
        "destination_country_id",
        "stops",
        "equipment",
        "route_count",
    )
)
print("fact_air_route:", fact_air_route.count())
fact_air_route.write.mode("overwrite").parquet(f"{REFINED}/fact_air_route")



fact_air_route: 66707


In [7]:
# Rutas salientes y entrantes por aeropuerto (distintas por aerolínea-destino/origen)
salientes = (
    routes_clean.groupBy(F.col("source_airport_id").alias("airport_id"))
    .agg(
        F.countDistinct("airline_id","dest_airport_id").alias("direct_routes_out"),
        F.countDistinct("dest_airport_id").alias("direct_destinations"),
        F.collect_set("airline_id").alias("al_out"),
    )
)
entrantes = (
    routes_clean.groupBy(F.col("dest_airport_id").alias("airport_id"))
    .agg(
        F.countDistinct("airline_id","source_airport_id").alias("direct_routes_in"),
        F.collect_set("airline_id").alias("al_in"),
    )
)
# Países destino directos por aeropuerto
dest_pais = (
    routes_clean
    .join(airports_clean.select(F.col("airport_id").alias("dest_airport_id"),
                                F.col("country_id").alias("dest_country_id")),
          "dest_airport_id", "left")
    .groupBy(F.col("source_airport_id").alias("airport_id"))
    .agg(F.countDistinct("dest_country_id").alias("direct_countries"))
)

fact_airport_connectivity = (
    airports_clean.select("airport_id")
    .join(salientes, "airport_id", "left")
    .join(entrantes, "airport_id", "left")
    .join(dest_pais, "airport_id", "left")
    .withColumn("airlines_count",
        F.size(F.array_distinct(F.concat(F.coalesce("al_out", F.array()),
                                         F.coalesce("al_in", F.array())))))
    .fillna({"direct_routes_out":0,"direct_routes_in":0,"direct_destinations":0,"direct_countries":0})
    .withColumn("connectivity_score",
        (F.col("direct_routes_out") + F.col("direct_routes_in")
         + F.col("direct_destinations")*F.lit(2)
         + F.col("direct_countries")*F.lit(5)
         + F.col("airlines_count")*F.lit(3)).cast("double"))
    .select("airport_id","direct_routes_out","direct_routes_in","direct_destinations",
            "direct_countries","airlines_count","connectivity_score")
)
print("fact_airport_connectivity:", fact_airport_connectivity.count())
fact_airport_connectivity.write.mode("overwrite").parquet(f"{REFINED}/fact_airport_connectivity")



fact_airport_connectivity: 8264


In [8]:
for t in ["dim_country","dim_airport","dim_airline","fact_air_route","fact_airport_connectivity"]:
    df = spark.read.parquet(f"{REFINED}/{t}")
    print(f"{t:28} filas={df.count():>7}  columnas={len(df.columns)}")

print("\nTop aeropuertos por conectividad:")
(spark.read.parquet(f"{REFINED}/fact_airport_connectivity")
    .join(spark.read.parquet(f"{REFINED}/dim_airport").select("airport_id","airport_name","iata_code"),
          "airport_id","left")
    .orderBy(F.desc("connectivity_score"))
    .select("iata_code","airport_name","direct_destinations","direct_countries","airlines_count","connectivity_score")
    .show(10, truncate=False))



dim_country                  filas=    314  columnas=4
dim_airport                  filas=   8264  columnas=9
dim_airline                  filas=   6162  columnas=6
fact_air_route               filas=  66707  columnas=8
fact_airport_connectivity    filas=   8264  columnas=7

Top aeropuertos por conectividad:
+---------+------------------------------------------------+-------------------+----------------+--------------+------------------+
|iata_code|airport_name                                    |direct_destinations|direct_countries|airlines_count|connectivity_score|
+---------+------------------------------------------------+-------------------+----------------+--------------+------------------+
|ATL      |Hartsfield Jackson Atlanta International Airport|217                |43              |38            |2589.0            |
|CDG      |Charles de Gaulle International Airport         |237                |105             |109           |2367.0            |
|FRA      |Frankfurt am Main A